In [81]:
from threading import Timer
from flask import Flask, send_from_directory

import dash
from dash import dcc, html
from dash.dependencies import Input, Output, State
import dash_leaflet as dl
import dash_bootstrap_components as dbc
 
from functions import * # in __innit__.py __all__ = ["filename.py", "otherfile.py",...]

In [82]:
UPLOAD_DIRECTORY = "/Frontend/app_uploaded_files"

#if not os.path.exists(UPLOAD_DIRECTORY):
#    os.makedirs(UPLOAD_DIRECTORY)

# Normally, Dash creates its own Flask server internally. By creating our own,
# we can create a route for downloading files directly:
server = Flask(__name__)

@server.route("/download/<path:path>")
def download(path):
    """Serve a file from the upload directory."""
    return send_from_directory(UPLOAD_DIRECTORY, path, as_attachment=True)

In [83]:
#external JavaScript files
external_scripts = []

#external CSS stylesheets
external_stylesheets = [
    dbc.themes.BOOTSTRAP,
]

# Create the app
app = dash.Dash(__name__, 
                external_stylesheets=external_stylesheets,
                external_scripts=external_scripts,
                )

# Set the page title
app.title = "TrafficLightNavigation"
app.description = "An interactive map of optimal routes for a specific start and end location."

In [84]:
# Initial markers locations and statuses
marker_1 = [{"id": "start", "lat": 53.5533, "lon": 10},
 {"id": "end", "lat": 53.5, "lon": 10}]
 
current_pos = [{"id": "current_pos", "lat": None, "lon": None}]

traffic_lights = [
     {"id": 1, "lat": None, "lon": None, "status": "red"},
     {"id": 2, "lat": None, "lon": None, "status": "green"}
     ]

# lat: latitude (Breite)
# lon: longitude (Länge)

In [85]:
headline = dbc.Row([
        dbc.Col(html.H1("Real-Time Traffic Light Map"), className="text-center")
        ])

In [86]:
upload = dcc.Upload(
            id="upload-data",
            children=html.Div(["Drag and drop or click to select a file to upload."]), 
            style={
                "width": "100%",
                "height": "60px",
                "lineHeight": "60px",
                "borderWidth": "1px",
                "borderStyle": "dashed",
                "borderRadius": "5px",
                "textAlign": "center",
                "margin": "10px",
            },
            multiple=True,
        ),

modal_fileupload = dbc.Col(
    [
        dbc.Button("Laden", id="open-centered", className="mb-3", n_clicks=0),
        dbc.Modal(
            [
                dbc.ModalHeader(dbc.ModalTitle("Laden: diese Funktion könnt ihr erstmal ignorieren. Man kann erstmal nur Sachen hochladen aber damit nichts anfangen."), close_button=True),
                dbc.ModalBody([
                    html.Div(upload),
                    html.Div("Hochgeladene Dateien:"),
                    html.Div(id="file-list")]),
                dbc.ModalFooter(
                    dbc.Button(
                        "Fertig",
                        id="close-centered",
                        className="ms-auto",
                        n_clicks=0,
                    )
                ),
            ],
            id="modal-centered",
            centered=True,
            is_open=False,
        ),
    ]
)

In [87]:
# the searchbar with the search field for the start and end location
# and the button to start the search
searchbar = dbc.Row(
            [   
                dbc.Col(
                    [dbc.InputGroup(
                        [dbc.InputGroupText("Id's"),
                        dbc.Input(type="search", id="id_selection", placeholder="8,100,1720,...", value=""),
                        ],
                        className="mb-3",
                        )
                    ], width = "auto"
                ),

                dbc.Col(
                    dbc.Button(
                        "Suchen",id="id_button", color="primary", className="mb-3", n_clicks=0
                        ),
                width="auto"
                ),

                dbc.Col(
                    [dbc.InputGroup(
                        [dbc.InputGroupText("Start"),
                        dbc.Input(type="search", id="startpunkt", placeholder="Informatikum", value=""),
                        dbc.FormFeedback("", type="invalid", id="start_invalid"),
                        dbc.FormFeedback("", type="valid", id="start_valid"),
                        ], 
                        className="mb-3",
                        )
                    ], width = "auto"
                ),
            
                dbc.Col(
                    [dbc.InputGroup(
                        [dbc.InputGroupText("Ziel"),
                        dbc.Input(type="search", id="endpunkt", placeholder="Hauptbahnhof", value=""),
                        dbc.FormFeedback("", type="invalid", id="end_invalid"), 
                        dbc.FormFeedback("", type="valid", id="end_valid"),
                        ],
                        className="mb-3",
                        )  
                    ], width="auto"
                ),
            
                dbc.Col(
                    dbc.Button(
                        "Route",id="route", color="primary", className="mb-3", n_clicks=0
                        ),
                width="auto"
                ),

                modal_fileupload,
            ],  
            className="g-3 ms-auto flex-nowrap mt-3 mt-md-0", 
            align="center",
        )

In [88]:
# the navigation bar, there you it will "toggle/collapse" if the window is smaller 
# in this field you can find the searchbar
navbar = dbc.Navbar(
    dbc.Container(
        [
            html.A(
                # Use row and col to control vertical alignment of logo / brand
                dbc.Row(
                    [
                        dbc.Col(dbc.NavbarBrand("Navbar", className="ms-0")),
                    ],
                    align="center",
                    className="g-0",
                ),
                style={"textDecoration": "none"},
            ),
            dbc.NavbarToggler(id="navbar-toggler", n_clicks=0),
            dbc.Collapse(
                searchbar,
                id="navbar-collapse",
                is_open=False,
                navbar=True,
            ),
        ], fluid = True,
    ),
    color="secondary",
    dark=True, 
)

In [89]:
# it will show the map with the markers and the Overview
route_map = dbc.Row([
                dbc.Col(dl.Map(center=[53.5533, 10.0081], zoom=14, 
                    children=[
                        dl.TileLayer(),
                        dl.LayerGroup(id="marker-layer2"),  # This is where markers will be added
                        dl.LayerGroup(id="marker-layer"),
                        dl.ScaleControl(position="bottomleft", imperial=False), #shows the scala 
                        dl.FullScreenControl()
                        ], 
                    style={'width': '100%', 'height': '600px'})
                ),
                
                dbc.Col(children= [html.Div("Übersicht: "), html.Div(id="coordinates_start", children = ""), 
                           html.Div(id="coordinates_end", children="")],
                    style={'width': '100%', 'height': '600px'}
                    )
            ],
            className="g-1", #change Row gutter (distance between the Columns)
        )

In [90]:
# Main layout
app.layout = dbc.Container(children = [
    headline,
    navbar,
    route_map,
    ],
    fluid = True, # remove the margins
    style={'padding': 0}, # removes ALL the margins
    className="bg-light", # background color
)

In [91]:
# triggers if you click the button
# this callback will get the Input for the start and end adresses
# it will give back the coordinates and shows them in the map as a marker

@app.callback(
    Output(component_id="startpunkt", component_property="valid"), 
    Output("startpunkt", "invalid"),
    Output(component_id="endpunkt", component_property="valid"), 
    Output("endpunkt", "invalid"),
    Output("route", "n_clicks"),
    Output("coordinates_start", "children"),
    Output("coordinates_end", "children"),
    Output('marker-layer', 'children'),
    [Input("startpunkt","value"),
     Input("endpunkt","value"),
     Input("route", "n_clicks"),
     ]
)
def check_validity(starttext, endtext, n_clicks): 
    marker = [{"id": "start", "lat": 53.5975929, "lon": 9.9318831},
                  {"id": "end", "lat": 53.5524079, "lon": 10.0084222}]
    if n_clicks:
        if starttext or endtext: 
            is_valid_start = True
            is_valid_end = True
            start_lat, start_lon = geocoding_utils.return_coordinates(starttext)
            end_lat, end_lon = geocoding_utils.return_coordinates(endtext)
            if start_lat == None:
                is_valid_start = False
            else:
                marker[0].update({"lat": start_lat, "lon": start_lon})
            if end_lat == None:
                is_valid_end = False
            else:
                marker[1].update({"lat": end_lat, "lon": end_lon})
            return is_valid_start, not is_valid_start, is_valid_end, not is_valid_end, 0, f"{start_lat}\t{start_lon}", f"{end_lat}\t{end_lon}", get_objects.get_marker(marker)
    return False, False, False, False, 0, "", "", get_objects.get_marker(marker) #no input yet


In [92]:
# add callback for toggling the collapse on small screens
@app.callback(
    Output("navbar-collapse", "is_open"),
    [Input("navbar-toggler", "n_clicks")],
    [State("navbar-collapse", "is_open")],
)
def toggle_navbar_collapse(n, is_open):
    if n:
        return not is_open
    return is_open

In [93]:
# opens the window for the upload 
@app.callback(
    Output("modal-centered", "is_open"),
    [Input("open-centered", "n_clicks"), Input("close-centered", "n_clicks")],
    [State("modal-centered", "is_open")],
)
def toggle_modal(n1, n2, is_open):
    if n1 or n2:
        return not is_open
    return is_open

In [94]:
# upload data and show which files you added
@app.callback(
    Output("file-list", "children"),
    [Input("upload-data", "filename"), Input("upload-data", "contents")],
)
def update_output(uploaded_filenames, uploaded_file_contents):
    """Save uploaded files and regenerate the file list."""

    if uploaded_filenames is not None and uploaded_file_contents is not None:
        for name, data in zip(uploaded_filenames, uploaded_file_contents):
            upload_file.save_file(name, data)

    files = upload_file.uploaded_files()
    if len(files) == 0:
        return [html.Li("No files yet!")]
    else:
        #return [html.Li(upload_file.file_download_link(filename)) for filename in files]
        option = [{"label": f"{filename}", "value": i} for i,filename in enumerate(files)]
        filelist = dbc.RadioItems(options= option, value=False, id="radioitems-input",)
        #return [dbc.Nav(upload_file.file_load_link(filename)) for filename in files]
        return filelist

In [95]:
@app.callback(
    Output("marker-layer2","children"),
    [Input("id_button","n_clicks"),
    Input("id_selection","value")],   
)
def find_id_and_show_in_map(n_clicks, ids):
    things = Things.Things([8,100,1720])
    if n_clicks:
        if(ids != ""):
            ids_list = list(map(int, ids.split(",")))
            things = Things.Things(ids_list)
            observations = things.get_tl_observations(ids_list, n_obsrvtns=1, phnTime=True)
            coordinations = things.get_coordinates()
            return get_objects.get_marker_id(coordinations, observations)
        return get_objects.get_marker_id(things.get_coordinates(),things.get_tl_observations([8,100,1720], n_obsrvtns=1, phnTime=True))
    return get_objects.get_marker_id(things.get_coordinates(),things.get_tl_observations([8,100,1720], n_obsrvtns=1, phnTime=True))
        


In [96]:
# Run the app
if __name__ == '__main__':
   Timer(1, open_browser.open_browser).start()
   app.run_server(debug=False, port=1222)